# Universities, Airports & Cars — Benchmark Record Generator

Implements domain-specific pool builders, query generators, and NLG templates for three entity types — universities, airports, and automobile models — used in the Wikidata multihop benchmark.

**Produces:**
- `universities_country_year_buckets_intl_strict_year_v2` pool (cached JSON)
- `airports_buckets_ru_v4` pool (cached JSON)
- `cars_manufacturers_ru` pool (cached JSON)
- Generator functions: `generate_universities_example`, `generate_airports_example`, `generate_cars_example`

## Imports & Configuration

In [ ]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires the nbformat package.
from pathlib import Path
if "BenchmarkExample" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())


## 2. Shared Utilities

`_gold_limit` controls how many candidates are fetched from Wikidata per complexity level, providing a buffer above the required answer count `k`.

In [ ]:
import random
from typing import List, Tuple, Optional

def _gold_limit(k: int, complexity: str) -> int:
    """
    Return the maximum number of gold items to request from WDQS for a given complexity level.
    Provides a buffer above the required answer count `k` to ensure enough candidates are fetched.
    Previously, generators silently failed in the main loop because this helper was absent,
    causing repeated retries without progress.
    """
    k = int(k)
    c = str(complexity)
    if c in ("L1", "L2"):
        return max(k + 5, 12)
    if c == "L3":
        return max(k + 6, 15)
    if c == "L4":
        return max(k + 8, 24)
    if c == "L5":
        return max(k + 5, 12)
    return max(k + 5, 12)

## Domain: Universities

Queries universities (Wikidata Q3918) filtered by country and optional founding-year range.
Complexity levels range from any university in a country (L1) to universities in future years that cannot exist (L5).

### 3a. Configuration and country vocabulary

In [ ]:
Q_UNIVERSITY = "Q3918"  # university

UNI_COUNTRY_ROWS = [
    {"country_qid": "Q159", "country_ru": "Россия",          "macro_region_ru": "РФ"},
    {"country_qid": "Q30",  "country_ru": "США",             "macro_region_ru": "Северная Америка"},
    {"country_qid": "Q16",  "country_ru": "Канада",          "macro_region_ru": "Северная Америка"},
    {"country_qid": "Q145", "country_ru": "Великобритания",  "macro_region_ru": "Европа"},
    {"country_qid": "Q183", "country_ru": "Германия",        "macro_region_ru": "Европа"},
    {"country_qid": "Q142", "country_ru": "Франция",         "macro_region_ru": "Европа"},
    {"country_qid": "Q38",  "country_ru": "Италия",          "macro_region_ru": "Европа"},
    {"country_qid": "Q29",  "country_ru": "Испания",         "macro_region_ru": "Европа"},
    {"country_qid": "Q55",  "country_ru": "Нидерланды",      "macro_region_ru": "Европа"},
    {"country_qid": "Q39",  "country_ru": "Швейцария",       "macro_region_ru": "Европа"},
    {"country_qid": "Q36",  "country_ru": "Польша",          "macro_region_ru": "Европа"},
    {"country_qid": "Q40",  "country_ru": "Австрия",         "macro_region_ru": "Европа"},
    {"country_qid": "Q213", "country_ru": "Чехия",           "macro_region_ru": "Европа"},
    {"country_qid": "Q27",  "country_ru": "Ирландия",        "macro_region_ru": "Европа"},
    {"country_qid": "Q33",  "country_ru": "Финляндия",       "macro_region_ru": "Европа"},
]
universities_countries_df = pd.DataFrame(UNI_COUNTRY_ROWS)

### 3b. Pool building and bucket selection

In [ ]:
def _build_universities_bucket_pool() -> pd.DataFrame:
    candidate_ranges = {
        "L1": [(None, None)],
        "L2": [(1800, 2025), (1850, 2025), (1900, 2025), (1950, 2025)],
        "L3": [(1800, 1999), (1850, 1999), (1900, 1999), (1950, 2025)],
        "L4": [(1600, 1899), (1700, 1899), (1800, 1949)],
        "L5": [(2500, 2500)],
    }
    need_k = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}
    rows = []

    for row in UNI_COUNTRY_ROWS:
        cqid = row["country_qid"]
        cru = row["country_ru"]
        macro = row["macro_region_ru"]

        for complexity, ranges in candidate_ranges.items():
            k = need_k[complexity]
            probe_limit = max(k, 6) if complexity != "L5" else 2

            for y1, y2 in ranges:
                where = [f"?uni wdt:P17 wd:{cqid} ."]
                if y1 is not None:
                    where += [
                        "?uni wdt:P571 ?inception .",
                        "BIND(YEAR(?inception) AS ?yy) .",
                        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
                    ]
                try:
                    _, items = select_items_with_label_ru_en(
                        Q_UNIVERSITY,
                        where,
                        limit=probe_limit,
                        item_var="uni",
                        use_subclass_closure=True,
                    )
                    n = len(items)
                except Exception:
                    continue

                ok = (
                    (complexity in ("L1", "L2") and n >= 5) or
                    (complexity == "L3" and n >= 4) or
                    (complexity == "L4" and n >= 1) or
                    (complexity == "L5" and n == 0)
                )
                if ok:
                    rows.append({
                        "complexity": complexity,
                        "country_qid": cqid,
                        "country_ru": cru,
                        "macro_region_ru": macro,
                        "y1": y1 if y1 is not None else "",
                        "y2": y2 if y2 is not None else "",
                        "probe_count": n,
                    })

    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

universities_bucket_pool_df = load_or_build_pool_safe(
    "universities_country_year_buckets_intl_strict_year_v2",
    _build_universities_bucket_pool,
)

def _pick_uni_bucket(complexity: str, rng: random.Random):
    df = universities_bucket_pool_df
    if isinstance(df, pd.DataFrame) and len(df) > 0:
        sub = df[df["complexity"] == complexity]
        if len(sub) > 0:
            row = sub.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
            y1 = row["y1"]
            y2 = row["y2"]
            y1 = None if y1 == "" or pd.isna(y1) else int(y1)
            y2 = None if y2 == "" or pd.isna(y2) else int(y2)
            return {
                "country_qid": str(row["country_qid"]),
                "country_ru": str(row["country_ru"]),
                "macro_region_ru": str(row["macro_region_ru"]),
                "y1": y1,
                "y2": y2,
            }

    base = rng.choice(UNI_COUNTRY_ROWS)
    if complexity == "L1":
        y1, y2 = None, None
    elif complexity == "L2":
        y1, y2 = rng.choice([(1800, 2025), (1850, 2025), (1900, 2025), (1950, 2025)])
    elif complexity == "L3":
        y1, y2 = rng.choice([(1800, 1999), (1850, 1999), (1900, 1999), (1950, 2025)])
    elif complexity == "L4":
        y1, y2 = rng.choice([(1600, 1899), (1700, 1899), (1800, 1949)])
    elif complexity == "L5":
        y1, y2 = 2500, 2500
    else:
        raise ValueError(f"Unknown complexity: {complexity}")
    return {**base, "y1": y1, "y2": y2}

### 3c. SPARQL query and NLG templates

In [ ]:
def run_universities_country_query(
    country_qid: str,
    y1: Optional[int],
    y2: Optional[int],
    limit: int = 60,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    where = [f"?uni wdt:P17 wd:{country_qid} ."]
    if y1 is not None:
        where += [
            "?uni wdt:P571 ?inception .",
                        "BIND(YEAR(?inception) AS ?yy) .",
                        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
        ]

    sparql, items = select_items_with_label_ru_en(
        Q_UNIVERSITY,
        where,
        limit=limit,
        item_var="uni",
        use_subclass_closure=True,
    )
    return sparql, items, where

def _nlg_uni_country_ru(country_ru: str, macro_region_ru: str, y1: Optional[int], y2: Optional[int], k: int, complexity: str) -> str:
    if y1 is None:
        return f"Перечисли {k} университетов страны «{country_ru}»."
    if complexity == "L4":
        return (f"Перечисли {k} университетов страны «{country_ru}» ({macro_region_ru}), "
                f"которые были основаны в период {y1}–{y2}. Если таких меньше {k}, перечисли все, сколько найдётся.")
    if complexity == "L5":
        return f"Перечисли {k} университетов страны «{country_ru}», основанных в период {y1}–{y2}."
    return f"Перечисли {k} университетов страны «{country_ru}», которые были основаны в период {y1}–{y2}."

### 3d. Example generator

In [ ]:
ADV_TEMPLATE_PROB_UNI = 0.0  # Disable advanced university templates for stable bucket filling

def generate_universities_example_default(complexity: str, idx: int, rng: random.Random, max_attempts: int = 40) -> BenchmarkExample:
    for _ in range(max_attempts):
        bucket = _pick_uni_bucket(complexity, rng)
        country_qid = bucket["country_qid"]
        country_ru = bucket["country_ru"]
        macro_region_ru = bucket["macro_region_ru"]
        y1 = bucket["y1"]
        y2 = bucket["y2"]

        if complexity == "L1":
            k = 5
        elif complexity == "L2":
            k = 5
        elif complexity == "L3":
            k = 4
        elif complexity == "L4":
            k = 3
        elif complexity == "L5":
            k = 3
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_universities_country_query(
            country_qid,
            y1,
            y2,
            limit=_gold_limit(k, complexity),
        )
        n = len(items)

        if complexity in ("L1", "L2") and n < k:
            continue
        if complexity == "L3" and n < k:
            continue
        if complexity == "L4" and n == 0:
            continue
        if complexity == "L5" and n != 0:
            continue

        ask = build_ask_validator(Q_UNIVERSITY, where, item_var="uni")
        return BenchmarkExample(
            id=f"universities_{complexity.lower()}_{idx:04d}",
            domain="universities",
            complexity=complexity,
            query_text_ru=_nlg_uni_country_ru(country_ru, macro_region_ru, y1, y2, k, complexity),
            constraints={
                "country_qid": country_qid,
                "country_ru": country_ru,
                "macro_region_ru": macro_region_ru,
                "y1": y1,
                "y2": y2,
            },
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=False,
            template_id="universities_country_year_fast",
            template_family="default",
            gold_truncated=len(items) >= _gold_limit(k, complexity),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(f"Universities default:{complexity} failed after {max_attempts} attempts")

def generate_universities_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 20) -> BenchmarkExample:
    raise RuntimeError("Universities advanced disabled for speed/stability")

def generate_universities_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 40) -> BenchmarkExample:
    return generate_universities_example_default(complexity, idx, rng, max_attempts=max_attempts)

## Domain: Airports

Queries airports (Wikidata Q1248784) filtered by country, optional IATA/ICAO code presence, and city population threshold.
Complexity levels range from any airport in a country (L1) to airports in impossibly large cities (L5).

### 4a. Configuration and country vocabulary

In [ ]:
Q_AIRPORT = ensure_qid("аэропорт", fallback_qid="Q1248784")
Q_CITY = "Q515"

AIRPORT_COUNTRY_ROWS = [
    {"country_qid": "Q159", "country_ru": "Россия"},
    {"country_qid": "Q30",  "country_ru": "США"},
    {"country_qid": "Q16",  "country_ru": "Канада"},
    {"country_qid": "Q145", "country_ru": "Великобритания"},
    {"country_qid": "Q183", "country_ru": "Германия"},
    {"country_qid": "Q142", "country_ru": "Франция"},
    {"country_qid": "Q38",  "country_ru": "Италия"},
    {"country_qid": "Q29",  "country_ru": "Испания"},
    {"country_qid": "Q55",  "country_ru": "Нидерланды"},
    {"country_qid": "Q17",  "country_ru": "Япония"},
    {"country_qid": "Q408", "country_ru": "Австралия"},
    {"country_qid": "Q155", "country_ru": "Бразилия"},
]

_AIRPORT_QUERY_CACHE: Dict[Tuple[str, int, bool, int], Tuple[str, List[Tuple[str, str]], List[str]]] = {}

### 4b. SPARQL query helper and pool building

In [ ]:
def run_airports_query(country_qid: str, pop_min: int = 0, require_code: bool = False, limit: int = 60) -> Tuple[str, List[Tuple[str,str]], List[str]]:
    key = (str(country_qid), int(pop_min), bool(require_code), int(limit))
    if key in _AIRPORT_QUERY_CACHE:
        return _AIRPORT_QUERY_CACHE[key]

    where = [
        f"?item wdt:P17 wd:{country_qid} .",
    ]
    if require_code:
        where.append("{ ?item wdt:P238 ?code . } UNION { ?item wdt:P239 ?code . }")

    if int(pop_min) > 0:
        where.extend([
            "OPTIONAL {",
            "  { ?item wdt:P931 ?city . ?city wdt:P31/wdt:P279* wd:Q515 . }",
            "  UNION",
            "  { ?item wdt:P131 ?city . ?city wdt:P31/wdt:P279* wd:Q515 . }",
            "  UNION",
            "  { ?item wdt:P276 ?city . ?city wdt:P31/wdt:P279* wd:Q515 . }",
            "  ?city wdt:P1082 ?pop .",
            "}",
            "BIND(COALESCE(?pop, 0) AS ?p) .",
            f"FILTER(?p >= {int(pop_min)}) .",
        ])

    sparql, items = select_items_with_label_ru_en(
        Q_AIRPORT,
        where,
        limit=limit,
        item_var="item",
        use_subclass_closure=True,
    )
    res = (sparql, items, where)
    _AIRPORT_QUERY_CACHE[key] = res
    return res

def build_airports_bucket_pool() -> pd.DataFrame:
    rows = []

    def add_bucket(complexity: str, country_qid: str, country_ru: str, pop_min: int, require_code: bool, n_items: int):
        rows.append({
            "complexity": complexity,
            "country_qid": country_qid,
            "country_ru": country_ru,
            "pop_min": int(pop_min),
            "require_code": int(require_code),
            "n_items": int(n_items),
        })

    for crow in AIRPORT_COUNTRY_ROWS:
        c_qid = crow["country_qid"]
        c_ru = crow["country_ru"]

        _, items, _ = run_airports_query(c_qid, pop_min=0, require_code=False, limit=30)
        if len(items) >= 5:
            add_bucket("L1", c_qid, c_ru, 0, False, len(items))

        _, items, _ = run_airports_query(c_qid, pop_min=0, require_code=True, limit=30)
        if len(items) >= 5:
            add_bucket("L2", c_qid, c_ru, 0, True, len(items))

        for pop_min in (200_000, 500_000, 1_000_000):
            _, items, _ = run_airports_query(c_qid, pop_min=pop_min, require_code=False, limit=30)
            if len(items) >= 5:
                add_bucket("L3", c_qid, c_ru, pop_min, False, len(items))

        for pop_min in (2_000_000, 3_000_000, 5_000_000):
            _, items, _ = run_airports_query(c_qid, pop_min=pop_min, require_code=True, limit=20)
            if len(items) >= 1:
                add_bucket("L4", c_qid, c_ru, pop_min, True, len(items))

        _, items, _ = run_airports_query(c_qid, pop_min=50_000_000, require_code=True, limit=10)
        if len(items) == 0:
            add_bucket("L5", c_qid, c_ru, 50_000_000, True, 0)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("airports bucket pool is empty")
    return df

airports_buckets_df = load_or_build_pool_safe(
    "airports_buckets_ru_v4",
    build_airports_bucket_pool,
)

def pick_airports_bucket(complexity: str, rng: random.Random) -> Dict[str, Any]:
    if not isinstance(airports_buckets_df, pd.DataFrame) or len(airports_buckets_df) == 0:
        raise RuntimeError("airports_buckets_df is empty")

    sub = airports_buckets_df[airports_buckets_df["complexity"] == complexity]
    if len(sub) == 0:
        raise RuntimeError(f"no airports buckets for {complexity}")

    row = sub.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return {
        "country_qid": str(row["country_qid"]),
        "country_ru": str(row["country_ru"]),
        "pop_min": int(row["pop_min"]),
        "require_code": bool(int(row["require_code"])),
        "n_items": int(row["n_items"]),
    }

### 4c. NLG templates and example generator

In [ ]:
def nlg_airports_ru(complexity: str, country_ru: str, pop_min: int, require_code: bool, k: int) -> str:
    if complexity == "L1":
        return f"Назови {k} аэропортов страны «{country_ru}»."
    if complexity == "L2":
        return f"Назови {k} аэропортов страны «{country_ru}», у которых указан код IATA или ICAO."
    if complexity == "L3":
        return f"Назови {k} аэропортов страны «{country_ru}», расположенных в городах с населением не меньше {pop_min} человек."
    if complexity == "L4":
        return f"Перечисли {k} аэропортов страны «{country_ru}», у которых указан код IATA или ICAO и которые расположены в городах с населением не меньше {pop_min} человек. Если таких меньше {k}, перечисли все, сколько есть."
    if complexity == "L5":
        return f"Назови {k} аэропортов страны «{country_ru}», у которых указан код IATA или ICAO и которые расположены в городах с населением не меньше {pop_min} человек. Если таких нет, так и напиши."
    raise ValueError(f"Unknown complexity: {complexity}")

ADV_TEMPLATE_PROB_AIRPORTS = 0.0

def generate_airports_example_default(complexity: str, idx: int, rng: random.Random, max_attempts: int = 25) -> BenchmarkExample:
    template_id = "airports_bucket_country_code_population"
    for _ in range(max_attempts):
        bucket = pick_airports_bucket(complexity, rng)

        if complexity in ("L1", "L2", "L3"):
            k = 5
        elif complexity == "L4":
            k = 12
        elif complexity == "L5":
            k = 5
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_airports_query(
            bucket["country_qid"],
            pop_min=bucket["pop_min"],
            require_code=bucket["require_code"],
            limit=_gold_limit(k, complexity),
        )
        n = len(items)

        if complexity in ("L1", "L2", "L3") and n < k:
            continue
        if complexity == "L4" and n == 0:
            continue
        if complexity == "L5" and n != 0:
            continue

        ask = build_ask_validator(Q_AIRPORT, where, item_var="item")
        return BenchmarkExample(
            id=f"airports_{complexity.lower()}_{idx:04d}",
            domain="airports",
            complexity=complexity,
            query_text_ru=nlg_airports_ru(complexity, bucket["country_ru"], bucket["pop_min"], bucket["require_code"], k),
            constraints={
                "country_qid": bucket["country_qid"],
                "country_ru": bucket["country_ru"],
                "pop_min": bucket["pop_min"],
                "require_code": bucket["require_code"],
            },
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=False,
            template_id=template_id,
            template_family="default",
            gold_truncated=len(items) >= _gold_limit(k, complexity),
            ask_validator_sparql=ask,
        )
    raise RuntimeError(f"Airports default:{complexity} failed after {max_attempts} attempts")

def generate_airports_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 1) -> BenchmarkExample:
    raise RuntimeError("airports advanced disabled")

def generate_airports_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 25) -> BenchmarkExample:
    return generate_airports_example_default(complexity, idx, rng, max_attempts=max_attempts)

## Domain: Cars

Queries automobile models (Wikidata Q3231690) filtered by manufacturer country, optional founding-year range, and specific manufacturer.
Complexity levels range from any car model by country (L1) to two-country cross queries (L4) and impossible future-year queries (L5).

### 5a. Configuration and country vocabulary

In [ ]:
Q_CAR_MODEL = ensure_qid("модель автомобиля", fallback_qid="Q3231690")  # automobile model

car_manufacturers_df = load_or_build_pool_safe(
    "cars_manufacturers_ru",
    lambda: build_value_pool_ru(Q_CAR_MODEL, "P176", "manu", limit=240),  # manufacturer
)

CAR_COUNTRY_ROWS = [
    {"country_qid": "Q17",  "country_ru": "Япония"},
    {"country_qid": "Q183", "country_ru": "Германия"},
    {"country_qid": "Q30",  "country_ru": "США"},
    {"country_qid": "Q142", "country_ru": "Франция"},
    {"country_qid": "Q38",  "country_ru": "Италия"},
    {"country_qid": "Q884", "country_ru": "Южная Корея"},
    {"country_qid": "Q148", "country_ru": "Китай"},
    {"country_qid": "Q145", "country_ru": "Великобритания"},
]

CAR_YEAR_RANGES_L2 = [(1960, 1979), (1980, 1999), (2000, 2020)]
CAR_YEAR_RANGES_L3 = [(1960, 1979), (1980, 1999), (2000, 2020)]
CAR_YEAR_RANGES_L4 = [(1960, 1999), (1980, 2020), (1950, 1985)]

def pick_car_country(rng: random.Random) -> Tuple[str, str]:
    row = rng.choice(CAR_COUNTRY_ROWS)
    return row["country_qid"], row["country_ru"]

### 5b. SPARQL WHERE-clause builders

In [ ]:
def _car_country_where(country_qid: str) -> List[str]:
    return [
        "?item wdt:P176 ?manu .",
        "OPTIONAL { ?manu (wdt:P17|wdt:P495) ?manu_country . }",
        "OPTIONAL { ?item (wdt:P495|wdt:P17) ?item_country . }",
        f"FILTER((BOUND(?manu_country) && ?manu_country = wd:{country_qid}) || (BOUND(?item_country) && ?item_country = wd:{country_qid})) .",
    ]

def _car_country_or_where(country1_qid: str, country2_qid: str) -> List[str]:
    return [
        "?item wdt:P176 ?manu .",
        "OPTIONAL { ?manu (wdt:P17|wdt:P495) ?manu_country . }",
        "OPTIONAL { ?item (wdt:P495|wdt:P17) ?item_country . }",
        f"FILTER((BOUND(?manu_country) && (?manu_country = wd:{country1_qid} || ?manu_country = wd:{country2_qid})) || (BOUND(?item_country) && (?item_country = wd:{country1_qid} || ?item_country = wd:{country2_qid}))) .",
    ]

def _car_year_where_strict(y1: int, y2: int) -> List[str]:
    return [
        "?item wdt:P571 ?inception .",
        "BIND(YEAR(?inception) AS ?yy) .",
        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
    ]

def _car_manufacturer_where(manu_qid: str) -> List[str]:
    return [
        f"?item wdt:P176 wd:{manu_qid} .",
    ]

CAR_BRAND_GENERIC_STOPWORDS = {
    "motor", "motors", "company", "corporation", "corp", "group", "auto",
    "automobile", "automobiles", "cars", "car", "co", "ltd", "inc", "ag", "gmbh",
    "spa", "s.p.a", "limited", "holding", "holdings"
}

CAR_MANUFACTURER_BRAND_HINTS = {
    "ford motor company": ["ford"],
    "toyota motor corporation": ["toyota"],
    "volkswagen ag": ["volkswagen"],
    "hyundai motor company": ["hyundai"],
    "honda motor company": ["honda"],
    "nissan motor": ["nissan"],
    "renault": ["renault"],
    "peugeot": ["peugeot"],
    "citroen": ["citroen", "citroën"],
    "fiat": ["fiat"],
    "bmw": ["bmw"],
    "bayerische motoren werke": ["bmw"],
    "mercedes benz group": ["mercedes-benz", "mercedes benz"],
    "mercedes benz": ["mercedes-benz", "mercedes benz"],
}

CAR_MANUFACTURER_BRAND_BLACKLIST = {
    "ford": ["lincoln", "mercury", "mercedes-benz", "mercedes benz"],
    "bmw": ["mini", "rolls-royce", "rolls royce"],
    "hyundai": ["kia", "genesis"],
    "toyota": ["lexus", "daihatsu", "hino"],
}

def _car_norm_brand_text(s: str) -> str:
    s = (s or "").strip().lower()
    for a, b in [
        ("ё", "е"), ("-", " "), ("/", " "), ("&", " and "),
        ("(", " "), (")", " "), (",", " "), (".", " "), ("'", " "), ('"', ' '),
    ]:
        s = s.replace(a, b)
    return " ".join(s.split())

def _car_brand_hints_for_manufacturer(manu_label: str) -> List[str]:
    raw = _car_norm_brand_text(manu_label)
    if not raw:
        return []

    hints: List[str] = []
    for key, vals in CAR_MANUFACTURER_BRAND_HINTS.items():
        if key in raw:
            hints.extend(vals)

    tokens = [t for t in raw.split() if t not in CAR_BRAND_GENERIC_STOPWORDS and len(t) >= 3]
    if tokens:
        hints.append(tokens[0])
        if len(tokens) >= 2:
            hints.append(" ".join(tokens[:2]))
    hints.append(raw)

    out: List[str] = []
    seen = set()
    for h in hints:
        h = _car_norm_brand_text(h)
        if h and h not in seen:
            seen.add(h)
            out.append(h)
    return out

def _car_brand_blacklist_for_manufacturer(manu_label: str) -> List[str]:
    hints = _car_brand_hints_for_manufacturer(manu_label)
    out: List[str] = []
    seen = set()
    for h in hints:
        for bad in CAR_MANUFACTURER_BRAND_BLACKLIST.get(h, []):
            bad = _car_norm_brand_text(bad)
            if bad and bad not in seen:
                seen.add(bad)
                out.append(bad)
    return out

def _car_brand_where_for_manufacturer(manu_label: str) -> List[str]:
    hints = _car_brand_hints_for_manufacturer(manu_label)
    blacklist = _car_brand_blacklist_for_manufacturer(manu_label)
    if not hints:
        return []

    contains_any_hint = " || ".join([f'CONTAINS(?brand_label_lc, "{h}")' for h in hints])
    not_blacklisted = " && ".join([f'!CONTAINS(?brand_label_lc, "{b}")' for b in blacklist]) or "true"

    return [
        "?item wdt:P1716 ?brand .",
        "OPTIONAL { ?brand rdfs:label ?brand_ru FILTER(LANG(?brand_ru)='ru') }",
        "OPTIONAL { ?brand rdfs:label ?brand_en FILTER(LANG(?brand_en)='en') }",
        "BIND(LCASE(COALESCE(STR(?brand_ru), STR(?brand_en), "")) AS ?brand_label_lc) .",
        f"FILTER(({contains_any_hint}) && ({not_blacklisted})) .",
    ]

### 5c. SPARQL query runners and manufacturer candidate finder

In [ ]:
def _run_car_models_where(where_lines: List[str], limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    sparql, items = select_items_with_label_ru_en(
        Q_CAR_MODEL,
        where_lines,
        limit=limit,
        item_var="item",
        use_subclass_closure=True,
    )
    return sparql, items, where_lines

def run_car_models_country_query(country_qid: str, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_where(country_qid),
    ], limit=limit)

def run_car_models_country_year_query(country_qid: str, y1: int, y2: int, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_where(country_qid),
        *_car_year_where_strict(y1, y2),
    ], limit=limit)

def run_car_models_country_year_manufacturer_query(country_qid: str, manu_qid: str, manu_label: str, y1: int, y2: int, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_where(country_qid),
        *_car_manufacturer_where(manu_qid),
        *_car_brand_where_for_manufacturer(manu_label),
        *_car_year_where_strict(y1, y2),
    ], limit=limit)

def run_car_models_two_countries_year_query(country1_qid: str, country2_qid: str, y1: int, y2: int, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_or_where(country1_qid, country2_qid),
        *_car_year_where_strict(y1, y2),
    ], limit=limit)

_CAR_MANUFACTURER_CANDIDATES_CACHE: Dict[Tuple[str, int, int, int, int], List[Tuple[str, str, int]]] = {}

def find_car_manufacturer_candidates(country_qid: str, y1: int, y2: int, min_models: int = 5, limit: int = 20) -> List[Tuple[str, str, int]]:
    key = (str(country_qid), int(y1), int(y2), int(min_models), int(limit))
    if key in _CAR_MANUFACTURER_CANDIDATES_CACHE:
        return _CAR_MANUFACTURER_CANDIDATES_CACHE[key]

    where = "\n      ".join([
        f"?item wdt:P31/wdt:P279* wd:{Q_CAR_MODEL} .",
        "?item wdt:P176 ?manu .",
        "OPTIONAL { ?manu (wdt:P17|wdt:P495) ?manu_country . }",
        "OPTIONAL { ?item (wdt:P495|wdt:P17) ?item_country . }",
        f"FILTER((BOUND(?manu_country) && ?manu_country = wd:{country_qid}) || (BOUND(?item_country) && ?item_country = wd:{country_qid})) .",
        "?item wdt:P571 ?inception .",
        "BIND(YEAR(?inception) AS ?yy) .",
        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
    ])

    sparql = f"""
    SELECT ?manu ?manuLabel (COUNT(DISTINCT ?item) AS ?cnt) WHERE {{
      {where}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
    }}
    GROUP BY ?manu ?manuLabel
    HAVING(COUNT(DISTINCT ?item) >= {int(min_models)})
    ORDER BY DESC(COUNT(DISTINCT ?item))
    LIMIT {int(limit)}
    """

    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int]] = []
    for r in rows:
        manu_qid = uri_to_qid(r.get("manu", ""))
        manu_label = r.get("manuLabel")
        cnt_raw = r.get("cnt", "0")
        try:
            cnt = int(float(cnt_raw))
        except Exception:
            cnt = 0
        if manu_qid and manu_label and cnt >= int(min_models):
            out.append((manu_qid, manu_label, cnt))

    _CAR_MANUFACTURER_CANDIDATES_CACHE[key] = out
    return out

### 5d. NLG templates and example generator

In [ ]:
def nlg_cars_l1_ru(country_ru: str, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителей из страны {country_ru}."

def nlg_cars_l2_ru(country_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителей из страны {country_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

def nlg_cars_l3_ru(country_ru: str, manu_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителя {manu_ru} из страны {country_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

def nlg_cars_l4_ru(country1_ru: str, country2_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителей из страны {country1_ru} или {country2_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

def nlg_cars_l5_ru(country_ru: str, manu_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителя {manu_ru} из страны {country_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

ADV_TEMPLATE_PROB_CARS = 0.0

def generate_cars_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    if complexity == "L1":
        template_id = "cars_country_only"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            sparql, items, where = run_car_models_country_query(c_qid, limit=_gold_limit(k, complexity))
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l1_ru(c_ru, k),
                constraints={"country_qid": c_qid, "country_ru": c_ru},
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L2":
        template_id = "cars_country_year"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            y1, y2 = rng.choice(CAR_YEAR_RANGES_L2)
            sparql, items, where = run_car_models_country_year_query(c_qid, y1, y2, limit=_gold_limit(k, complexity))
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l2_ru(c_ru, y1, y2, k),
                constraints={"country_qid": c_qid, "country_ru": c_ru, "y1": y1, "y2": y2},
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L3":
        template_id = "cars_country_year_manufacturer"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            y1, y2 = rng.choice(CAR_YEAR_RANGES_L3)
            candidates = find_car_manufacturer_candidates(c_qid, y1, y2, min_models=k, limit=20)
            if not candidates:
                continue

            top = candidates[:min(len(candidates), 8)]
            manu_qid, manu_ru, _ = rng.choice(top)
            sparql, items, where = run_car_models_country_year_manufacturer_query(
                c_qid, manu_qid, manu_ru, y1, y2, limit=_gold_limit(k, complexity)
            )
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l3_ru(c_ru, manu_ru, y1, y2, k),
                constraints={
                    "country_qid": c_qid,
                    "country_ru": c_ru,
                    "manufacturer_qid": manu_qid,
                    "manufacturer_ru": manu_ru,
                    "y1": y1,
                    "y2": y2,
                },
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L4":
        template_id = "cars_two_countries_year"
        k = 8
        for _ in range(max_attempts):
            c1_qid, c1_ru = pick_car_country(rng)
            c2_qid, c2_ru = pick_car_country(rng)
            if c1_qid == c2_qid:
                continue
            y1, y2 = rng.choice(CAR_YEAR_RANGES_L4)

            sparql, items, where = run_car_models_two_countries_year_query(
                c1_qid, c2_qid, y1, y2, limit=_gold_limit(k, complexity)
            )
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l4_ru(c1_ru, c2_ru, y1, y2, k),
                constraints={
                    "country1_qid": c1_qid,
                    "country1_ru": c1_ru,
                    "country2_qid": c2_qid,
                    "country2_ru": c2_ru,
                    "y1": y1,
                    "y2": y2,
                },
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L5":
        template_id = "cars_country_manufacturer_future_year_zero"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            seed_y1, seed_y2 = rng.choice(CAR_YEAR_RANGES_L3)
            candidates = find_car_manufacturer_candidates(c_qid, seed_y1, seed_y2, min_models=2, limit=20)
            if not candidates:
                continue

            manu_qid, manu_ru, _ = rng.choice(candidates[:min(len(candidates), 8)])
            y1, y2 = 2500, 2505
            sparql, items, where = run_car_models_country_year_manufacturer_query(
                c_qid, manu_qid, manu_ru, y1, y2, limit=_gold_limit(k, complexity)
            )
            if len(items) != 0:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l5_ru(c_ru, manu_ru, y1, y2, k),
                constraints={
                    "country_qid": c_qid,
                    "country_ru": c_ru,
                    "manufacturer_qid": manu_qid,
                    "manufacturer_ru": manu_ru,
                    "y1": y1,
                    "y2": y2,
                },
                requested_count=k,
                gold_answer_qids=[],
                gold_answer_labels_ru=[],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=False,
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    raise ValueError(f"Unknown complexity: {complexity}")

def generate_cars_example_default(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    return generate_cars_example(complexity, idx, rng, max_attempts=max_attempts)

def generate_cars_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    return generate_cars_example(complexity, idx, rng, max_attempts=max_attempts)

## 6. Finalization

Optional inter-domain pause (used when this notebook is called as part of a sequential pipeline run).

In [ ]:
if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))